In [42]:
import pandas as pd
import numpy as np
import optuna
import plotly.express as px

import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [43]:
best_params = {'learning_rate': 0.051179166440270964, 'grow_policy': 'depthwise', 
               'max_depth': 3, 'subsample': 0.9236126590824028, 
               'colsample_bytree': 0.8756451342494557, 'reg_alpha': 19.11532865442273, 
               'reg_lambda': 0.0006386455580355257, 'min_child_weight': 6, 
               'gamma': 0.011557488335842095, 'objective': 'reg:squarederror', 
               'tree_method': 'auto', 'eval_metric': 'rmse', 'n_estimators': 2500}

In [53]:
train_img_dict = np.load("/kaggle/input/property-valuation-03encodings/train_image_features_b4.npy", allow_pickle=True).item()
test_img_dict = np.load("/kaggle/input/property-valuation-03encodings/test_image_features_b4.npy", allow_pickle=True).item()

train_encoding_df = pd.DataFrame.from_dict(train_img_dict, orient="index").reset_index()
test_encoding_df = pd.DataFrame.from_dict(test_img_dict,orient='index').reset_index()

train_encoding_df = train_encoding_df.rename(columns={"index":"house_id"})
test_encoding_df  = test_encoding_df.rename(columns={"index":"house_id"})

model_1 = StandardScaler()
model_1.fit(train_encoding_df.drop(columns="house_id"))
std_arr = model_1.transform(train_encoding_df.drop(columns="house_id"))
std_arr_test = model_1.transform(test_encoding_df.drop(columns="house_id"))

pca = PCA(n_components=0.40, random_state=42)  # 40% varience => top 12 principal components
pca.fit(std_arr)
pca_arr = pca.transform(std_arr)
pca_arr_test = pca.transform(std_arr_test)

pca_df = pd.DataFrame(pca_arr)
pca_df_test = pd.DataFrame(pca_arr_test)
pca_df['house_id'] = train_encoding_df['house_id']
pca_df_test['house_id'] = test_encoding_df['house_id']
pca_df = pca_df.set_index("house_id").reset_index()
pca_df_test = pca_df_test.set_index("house_id").reset_index()

In [56]:
df = pd.read_csv("/kaggle/input/property-val-dataset/train.csv")
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

df['effective_built_year'] = df.apply(lambda x: x['yr_renovated'] if x['yr_renovated'] > 0 else x['yr_built'], axis=1)
df['house_age'] = df['sale_year'] - df['effective_built_year']

zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()
df['zip_index'] = df['zipcode'].map(zip_map)


df_test = pd.read_csv("/kaggle/input/property-val-dataset/test.csv")
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['sale_year'] = df_test['date'].dt.year
df_test['sale_month'] = df_test['date'].dt.month
df_test['day_of_month'] = df_test['date'].dt.day
df_test['day_of_week'] = df_test['date'].dt.dayofweek

df_test['effective_built_year'] = df_test.apply(lambda x: x['yr_renovated'] if x['yr_renovated'] > 0 else x['yr_built'], axis=1)
df_test['house_age'] = df_test['sale_year'] - df_test['effective_built_year']

#zip_map = df_test.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()
df_test['zip_index'] = df_test['zipcode'].map(zip_map)

/tmp/ipykernel_47/176837582.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()


In [61]:
final_df = df.merge(pca_df,how='outer',left_on = 'id',right_on='house_id')
final_df_test = df_test.merge(pca_df_test,how='outer',left_on = 'id',right_on='house_id')

cols_to_drop = ['id', 'date','zipcode',"house_id"]  

final_df.drop(columns=cols_to_drop,inplace=True)
final_df_test.drop(columns=cols_to_drop,inplace=True)

In [63]:
final_df_test.isnull().sum()

bedrooms                0
bathrooms               0
sqft_living             0
sqft_lot                0
floors                  0
waterfront              0
view                    0
condition               0
grade                   0
sqft_above              0
sqft_basement           0
yr_built                0
yr_renovated            0
lat                     0
long                    0
sqft_living15           0
sqft_lot15              0
sale_year               0
sale_month              0
day_of_month            0
day_of_week             0
effective_built_year    0
house_age               0
zip_index               0
0                       0
1                       0
2                       0
3                       0
4                       0
5                       0
6                       0
7                       0
8                       0
9                       0
10                      0
11                      0
dtype: int64

In [69]:
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(final_df.drop(columns='price'),final_df['price'])
y_pred = final_model.predict(final_df_test)

In [72]:
test_df = pd.read_csv("/kaggle/input/property-val-dataset/test.csv")
submission = pd.DataFrame({
    'id': test_df['id'],
    'price': y_pred })

submission.to_csv('submission_final.csv', index=False)

In [74]:
submission.isnull().sum()

id       0
price    0
dtype: int64